No `pandas` o segredo está na função `transform` logo após a agregação para criar `window functions`.

In [1]:
import pandas as pd
from pathlib import Path

In [4]:
FILE = Path('../../data/window_functions_salles.pq')
df = pd.read_parquet(FILE)
df.head(5)

,regiao,produto,mes,vendas,renda
0,norte,A,2026-01,152,166867.0
1,norte,A,2026-02,142,123694.0
2,norte,A,2026-03,156,139879.0
3,norte,A,2026-04,238,74886.0
4,norte,A,2026-05,171,188266.0


## Ranking

In [7]:
df['primeiro_dia_de_vendas_por_regiao'] = (
    df.groupby('regiao')['mes']
    .rank(method='dense', ascending=True)
)
df.head()

,regiao,produto,mes,vendas,renda,primeiro_dia_de_vendas_por_regiao
0,norte,A,2026-01,152,166867.0,1.0
1,norte,A,2026-02,142,123694.0,2.0
2,norte,A,2026-03,156,139879.0,3.0
3,norte,A,2026-04,238,74886.0,4.0
4,norte,A,2026-05,171,188266.0,5.0


## Agregações

In [14]:
df_agg = df.copy()
df_agg['media_vendas_regiao'] = df_agg.groupby('regiao')['vendas'].transform('mean')
df_agg['soma_vendas_regiao'] = df_agg.groupby('regiao')['vendas'].transform('sum')
df_agg['max_vendas_regiao'] = df_agg.groupby('regiao')['vendas'].transform('max')
df_agg['min_vendas_regiao'] = df_agg.groupby('regiao')['vendas'].transform('min')
df_agg['count_vendas_regiao'] = df_agg.groupby('regiao')['vendas'].transform('count')
df_agg['std_vendas_regiao'] = df_agg.groupby('regiao')['vendas'].transform('std')

COLS = [
    'regiao',
    'vendas',
    'media_vendas_regiao',
    'soma_vendas_regiao',
    'max_vendas_regiao',
    'min_vendas_regiao', 
    'count_vendas_regiao',
    'std_vendas_regiao'
]
df_agg[COLS].head()

,regiao,vendas,media_vendas_regiao,soma_vendas_regiao,max_vendas_regiao,min_vendas_regiao,count_vendas_regiao,std_vendas_regiao
0,norte,152,161.333333,1936,252,87,12,47.442469
1,norte,142,161.333333,1936,252,87,12,47.442469
2,norte,156,161.333333,1936,252,87,12,47.442469
3,norte,238,161.333333,1936,252,87,12,47.442469
4,norte,171,161.333333,1936,252,87,12,47.442469


## LEG e LEAD

In [16]:
vendas_mes = (
    df.groupby('mes')['vendas']
    .sum()
    .reset_index(name='sum_salles')
    .sort_values('mes')
)

vendas_mes['past_salles'] = vendas_mes['sum_salles'].shift(1)
vendas_mes['next_salles'] = vendas_mes['sum_salles'].shift(-1)

vendas_mes

,mes,sum_salles,past_salles,next_salles
0,2026-01,794,NaN,605.0
1,2026-02,605,794.0,761.0
2,2026-03,761,605.0,852.0
3,2026-04,852,761.0,875.0
4,2026-05,875,852.0,874.0
5,2026-06,874,875.0,NaN


## Cum Sum

In [18]:
df_cum = df.copy()
df_cum['acumulado'] = df_cum.groupby('regiao')['vendas'].cumsum()
df_cum['max_acumulado'] = df_cum.groupby('regiao')['vendas'].cummax()
df_cum['min_acumulado'] = df_cum.groupby('regiao')['vendas'].cummin()

df_cum.head()

,regiao,produto,mes,vendas,renda,primeiro_dia_de_vendas_por_regiao,acumulado,max_acumulado,min_acumulado
0,norte,A,2026-01,152,166867.0,1.0,152,152,152
1,norte,A,2026-02,142,123694.0,2.0,294,152,142
2,norte,A,2026-03,156,139879.0,3.0,450,156,142
3,norte,A,2026-04,238,74886.0,4.0,688,238,142
4,norte,A,2026-05,171,188266.0,5.0,859,238,142


## Médias Moveis

In [20]:
df_roll = df.copy()
df_roll['media_movel_3'] = df_roll.groupby('regiao')['vendas'].transform(lambda s: s.rolling(3).mean())
df_roll['soma_movel_3'] = df_roll.groupby('regiao')['vendas'].transform(lambda s: s.rolling(3).sum())
df_roll['max_movel_3'] = df_roll.groupby('regiao')['vendas'].transform(lambda s: s.rolling(3).max())
df_roll['min_movel_3'] = df_roll.groupby('regiao')['vendas'].transform(lambda s: s.rolling(3).min())
df_roll['std_movel_3'] = df_roll.groupby('regiao')['vendas'].transform(lambda s: s.rolling(3).std())

df_roll.head()

,regiao,produto,mes,vendas,renda,primeiro_dia_de_vendas_por_regiao,media_movel_3,soma_movel_3,max_movel_3,min_movel_3,std_movel_3
0,norte,A,2026-01,152,166867.0,1.0,NaN,NaN,NaN,NaN,NaN
1,norte,A,2026-02,142,123694.0,2.0,NaN,NaN,NaN,NaN,NaN
2,norte,A,2026-03,156,139879.0,3.0,150.000000,450.0,156.0,142.0,7.211103
3,norte,A,2026-04,238,74886.0,4.0,178.666667,536.0,238.0,142.0,51.858783
4,norte,A,2026-05,171,188266.0,5.0,188.333333,565.0,238.0,156.0,43.661577


## NTILE
Faz uma divisão de *n* partes nos seus dados, por exemplo se você possui 100 linhas e divide em 4 partes teremos 25 linhas em cada parte dado a coluna que você utilizou para divisão.

In [22]:
df_ntile = df.copy()
df_ntile['faixa_renda'] = (
    df_ntile.groupby('regiao')['renda']
    .transform(lambda s: pd.qcut(s, 4, labels=["1", "2", "3", "4"]))
)

df_ntile.head()

,regiao,produto,mes,vendas,renda,primeiro_dia_de_vendas_por_regiao,faixa_renda
0,norte,A,2026-01,152,166867.0,1.0,4
1,norte,A,2026-02,142,123694.0,2.0,3
2,norte,A,2026-03,156,139879.0,3.0,3
3,norte,A,2026-04,238,74886.0,4.0,2
4,norte,A,2026-05,171,188266.0,5.0,4
